In [ ]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.9 MB/s eta 0:00:00


In [ ]:
import json
import random
import time
from groq import Groq
from google.colab import userdata
api_key = userdata.get('YOUR_GROQ_API_KEY')

client = Groq(api_key=api_key)

SYSTEM_PROMPT = """
You are an expert MLOps router. Output strictly valid JSON.
Given an incident description, determine the tools to call, bottleneck, and risk level.
Allowed tools: [
    "query_servicenow_db", "fetch_network_logs", "restart_ec2_node",
    "trigger_anomaly_detection", "page_oncall"
]
Format: {"tools_to_call": ["tool1"], "predicted_bottleneck": "reason", "risk_level": "high|medium|low"}
"""

INCIDENT_TEMPLATES = [
    "High latency detected on primary database during a large collection drive.",
    "Network traffic anomaly detected on node {node_id} resembling a DDoS pattern.",
    "Asynchronous queue backup for profile picture uploads.",
    "Memory leak suspect in the Rescue Triage image processing pipeline."
]

def generate_synthetic_trace(incident_desc):
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Incident: {incident_desc}"}
            ],
            temperature=0.7,
            response_format={"type": "json_object"}
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"Error: {e}")
        return None

def build_dataset(num_samples=100):
    dataset = []

    for i in range(num_samples):
        incident = random.choice(INCIDENT_TEMPLATES).format(node_id=random.randint(100, 999))
        result = generate_synthetic_trace(incident)

        if result:
            dataset.append({
                "input_query": incident,
                "target_json": result
            })
            print(f"Generated {i+1}/{num_samples}")

        time.sleep(2)

    with open("synthetic_incident_routing.json", "w") as f:
        json.dump(dataset, f, indent=2)

if __name__ == "__main__":
    build_dataset(100)

Generated 1/100
Generated 2/100
Generated 3/100
Generated 4/100
Generated 5/100
Generated 6/100
Generated 7/100
Generated 8/100
Generated 9/100
Generated 10/100
Generated 11/100
Generated 12/100
Generated 13/100
Generated 14/100
Generated 15/100
Generated 16/100
Generated 17/100
Generated 18/100
Generated 19/100
Generated 20/100
Generated 21/100
Generated 22/100
Generated 23/100
Generated 24/100
Generated 25/100
Generated 26/100
Generated 27/100
Generated 28/100
Generated 29/100
Generated 30/100
Generated 31/100
Generated 32/100
Generated 33/100
Generated 34/100
Generated 35/100
Generated 36/100
Generated 37/100
Generated 38/100
Generated 39/100
Generated 40/100
Generated 41/100
Generated 42/100
Generated 43/100
Generated 44/100
Generated 45/100
Generated 46/100
Generated 47/100
Generated 48/100
Generated 49/100
Generated 50/100
Generated 51/100
Generated 52/100
Generated 53/100
Generated 54/100
Generated 55/100
Generated 56/100
Generated 57/100
Generated 58/100
Generated 59/100
Genera

In [ ]:
import json
import random
import time
from groq import Groq
from collections import Counter
from google.colab import userdata
api_key = userdata.get('YOUR_GROQ_API_KEY')
client = Groq(api_key=api_key)

SYSTEM_PROMPT = """
You are an expert MLOps router. Output strictly valid JSON.
Given an incident description, determine the tools to call, bottleneck, and risk level.
Allowed tools: [
    "query_servicenow_db", "fetch_network_logs", "restart_ec2_node",
    "trigger_anomaly_detection", "page_oncall"
]
Format: {"tools_to_call": ["tool1"], "predicted_bottleneck": "reason", "risk_level": "high|medium|low"}
"""

ALLOWED_TOOLS = {
    "query_servicenow_db", "fetch_network_logs", "restart_ec2_node",
    "trigger_anomaly_detection", "page_oncall"
}

# EXPANDED: 50+ incident templates (not just 4)
INCIDENT_TEMPLATES = [
    # Database issues
    "Database connection pool exhaustion on primary PostgreSQL during peak traffic surge.",
    "Primary database latency spike from 10ms to 2000ms detected on analytics cluster.",
    "Deadlock detected in account_transactions table, blocking payment processing.",
    "OOM error on MySQL primary node causing query evictions.",
    "Replication lag between primary and replica exceeding 5 seconds.",

    # Network issues
    "Network traffic anomaly on node {node_id} showing 100x baseline, resembling DDoS.",
    "BGP route flap causing intermittent connectivity to AWS region {node_id}.",
    "High packet loss (15%) detected on NIC eth0 on {node_id}.",
    "DNS resolution failures for internal service discovery affecting pod-to-pod communication.",
    "Network partition between data centers causing split-brain scenario.",

    # Application issues
    "Memory leak in Rescue AI Triage pipeline, heap size growing 5% per hour.",
    "Race condition in async profile picture upload causing duplicate entries.",
    "Goroutine leak in webhook consumer causing process memory exhaustion.",
    "CPU throttling detected on {node_id} due to noisy neighbor consuming 95% cores.",
    "GC pause spikes of 500ms freezing request handlers.",

    # Infrastructure issues
    "EC2 instance {node_id} failed health check, suspect faulty hardware.",
    "Disk space on /var/lib/docker nearly full (95%), container deployments failing.",
    "EBS volume performance degradation, IOPS dropping to 10% of provisioned.",
    "Kubernetes API server unresponsive, unable to schedule new pods.",
    "Load balancer health check failures on 30% of backend targets.",

    # Data pipeline issues
    "Kafka consumer lag exceeding 1 hour for payment events topic.",
    "Data warehouse ETL job timeout after 6 hours, data freshness SLA breached.",
    "Elasticsearch cluster in yellow state, shard reallocation hanging.",
    "S3 object upload bottleneck during large batch import job.",
    "Redis cache eviction rate spiking, key hit rate dropping to 60%.",
]

def validate_schema(json_obj):
    """Validate JSON matches expected schema"""
    if not isinstance(json_obj, dict):
        return False, "Not a dict"

    required_keys = {"tools_to_call", "predicted_bottleneck", "risk_level"}
    if not required_keys.issubset(json_obj.keys()):
        return False, f"Missing keys: {required_keys - set(json_obj.keys())}"

    # Validate tools_to_call
    if not isinstance(json_obj["tools_to_call"], list):
        return False, "tools_to_call not a list"

    if not all(tool in ALLOWED_TOOLS for tool in json_obj["tools_to_call"]):
        invalid_tools = [t for t in json_obj["tools_to_call"] if t not in ALLOWED_TOOLS]
        return False, f"Invalid tools: {invalid_tools}"

    if len(json_obj["tools_to_call"]) == 0:
        return False, "tools_to_call is empty"

    # Validate risk_level
    if json_obj["risk_level"] not in {"high", "medium", "low"}:
        return False, f"Invalid risk_level: {json_obj['risk_level']}"

    # Validate predicted_bottleneck
    if not isinstance(json_obj["predicted_bottleneck"], str):
        return False, "predicted_bottleneck not a string"

    if len(json_obj["predicted_bottleneck"]) < 10:
        return False, "predicted_bottleneck too short"

    if len(json_obj["predicted_bottleneck"]) > 500:
        return False, "predicted_bottleneck too long"

    return True, "Valid"

def generate_synthetic_trace(incident_desc, max_retries=3):
    """Generate with automatic retry + schema validation"""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": f"Incident: {incident_desc}"}
                ],
                temperature=0.7,
                response_format={"type": "json_object"}
            )

            result = json.loads(response.choices[0].message.content)
            is_valid, reason = validate_schema(result)

            if is_valid:
                return result
            else:
                print(f"  ❌ Invalid schema (attempt {attempt+1}/{max_retries}): {reason}")

        except json.JSONDecodeError as e:
            print(f"  ❌ JSON decode error (attempt {attempt+1}/{max_retries}): {e}")
        except Exception as e:
            print(f"  ❌ Error (attempt {attempt+1}/{max_retries}): {e}")

        if attempt < max_retries - 1:
            time.sleep(2 ** attempt)  # Exponential backoff

    return None

def build_dataset(num_samples=200):
    """Build synthetic dataset with quality checks"""
    dataset = []
    generated = 0
    failed = 0

    print(f"Generating {num_samples} synthetic training examples...\n")

    for i in range(num_samples):
        incident = random.choice(INCIDENT_TEMPLATES).format(
            node_id=random.randint(100, 999)
        )

        print(f"[{i+1}/{num_samples}] Incident: {incident[:60]}...")
        result = generate_synthetic_trace(incident)

        if result:
            dataset.append({
                "input_query": incident,
                "target_json": result
            })
            generated += 1
            print(f"  ✅ Success")
        else:
            failed += 1
            print(f"  ❌ Failed after retries")

        # Rate limiting
        time.sleep(1)

    print(f"\n=== Generation Summary ===")
    print(f"Generated: {generated}/{num_samples}")
    print(f"Failed: {failed}/{num_samples}")
    print(f"Success rate: {generated/num_samples*100:.1f}%")

    # Analyze dataset quality
    if dataset:
        analyze_dataset_quality(dataset)

    # Save with splits
    save_dataset_with_splits(dataset)

def analyze_dataset_quality(dataset):
    """Analyze and report dataset quality"""
    tool_dist = Counter()
    risk_dist = Counter()
    bottleneck_lengths = []

    for example in dataset:
        target = example["target_json"]

        for tool in target["tools_to_call"]:
            tool_dist[tool] += 1

        risk_dist[target["risk_level"]] += 1
        bottleneck_lengths.append(len(target["predicted_bottleneck"].split()))

    print(f"\n=== Dataset Quality Analysis ===")
    print(f"Total samples: {len(dataset)}")

    print(f"\nTool distribution:")
    for tool, count in tool_dist.most_common():
        pct = count / len(dataset) * 100
        print(f"  {tool}: {count} ({pct:.1f}%)")

    print(f"\nRisk level distribution:")
    for risk, count in risk_dist.most_common():
        pct = count / len(dataset) * 100
        print(f"  {risk}: {count} ({pct:.1f}%)")

    print(f"\nBottleneck description stats:")
    print(f"  Min length: {min(bottleneck_lengths)} words")
    print(f"  Max length: {max(bottleneck_lengths)} words")
    print(f"  Avg length: {sum(bottleneck_lengths)/len(bottleneck_lengths):.1f} words")

def save_dataset_with_splits(dataset, output_dir="data"):
    """Save dataset with train/val/test splits"""
    import os
    os.makedirs(output_dir, exist_ok=True)

    random.shuffle(dataset)

    train_size = int(0.8 * len(dataset))
    val_size = int(0.1 * len(dataset))

    train_set = dataset[:train_size]
    val_set = dataset[train_size:train_size + val_size]
    test_set = dataset[train_size + val_size:]

    print(f"\n=== Saving Dataset ===")
    for split_name, split_data in [
        ('train', train_set),
        ('val', val_set),
        ('test', test_set)
    ]:
        path = f"{output_dir}/{split_name}.jsonl"
        with open(path, 'w') as f:
            for example in split_data:
                f.write(json.dumps(example) + '\n')

        print(f"✅ Saved {split_name}: {len(split_data)} examples → {path}")

if __name__ == "__main__":
    build_dataset(num_samples=200)

Generating 200 synthetic training examples...

[1/200] Incident: Network partition between data centers causing split-brain s...
  ✅ Success
[2/200] Incident: Goroutine leak in webhook consumer causing process memory ex...
  ✅ Success
[3/200] Incident: Goroutine leak in webhook consumer causing process memory ex...
  ✅ Success
[4/200] Incident: GC pause spikes of 500ms freezing request handlers....
  ✅ Success
[5/200] Incident: Data warehouse ETL job timeout after 6 hours, data freshness...
  ✅ Success
[6/200] Incident: Network partition between data centers causing split-brain s...
  ✅ Success
[7/200] Incident: Goroutine leak in webhook consumer causing process memory ex...
  ✅ Success
[8/200] Incident: GC pause spikes of 500ms freezing request handlers....
  ✅ Success
[9/200] Incident: Replication lag between primary and replica exceeding 5 seco...
  ✅ Success
[10/200] Incident: GC pause spikes of 500ms freezing request handlers....
  ✅ Success
[11/200] Incident: Data warehouse ETL j

In [ ]:
import json
import time
from groq import Groq
from collections import defaultdict, Counter
import statistics
from google.colab import userdata
api_key = userdata.get('YOUR_GROQ_API_KEY')
client = Groq(api_key=api_key)

SYSTEM_PROMPT = """
You are an expert MLOps router. Output strictly valid JSON.
Given an incident description, determine the tools to call, bottleneck, and risk level.
Allowed tools: [
    "query_servicenow_db", "fetch_network_logs", "restart_ec2_node",
    "trigger_anomaly_detection", "page_oncall"
]
Format: {"tools_to_call": ["tool1"], "predicted_bottleneck": "reason", "risk_level": "high|medium|low"}
"""

ALLOWED_TOOLS = {
    "query_servicenow_db", "fetch_network_logs", "restart_ec2_node",
    "trigger_anomaly_detection", "page_oncall"
}

class ValidationResult:
    def __init__(self):
        self.total_samples = 0
        self.valid_json = 0
        self.schema_valid = 0
        self.tool_exact_match = 0
        self.risk_level_match = 0
        self.latencies = []
        self.errors = defaultdict(int)
        self.detailed_results = []
        self.per_tool_stats = defaultdict(lambda: {"correct": 0, "total": 0})
        self.per_risk_stats = defaultdict(lambda: {"correct": 0, "total": 0})

def validate_schema(json_obj):
    """Strictly validate JSON schema"""
    if not isinstance(json_obj, dict):
        return False, "Not a dict"

    required_keys = {"tools_to_call", "predicted_bottleneck", "risk_level"}
    if not required_keys.issubset(json_obj.keys()):
        return False, f"Missing keys: {required_keys - set(json_obj.keys())}"

    # Validate tools_to_call
    if not isinstance(json_obj["tools_to_call"], list):
        return False, "tools_to_call not a list"

    if len(json_obj["tools_to_call"]) == 0:
        return False, "tools_to_call is empty"

    if not all(tool in ALLOWED_TOOLS for tool in json_obj["tools_to_call"]):
        invalid = [t for t in json_obj["tools_to_call"] if t not in ALLOWED_TOOLS]
        return False, f"Invalid tools: {invalid}"

    # Validate risk_level
    if json_obj["risk_level"] not in {"high", "medium", "low"}:
        return False, f"Invalid risk_level: {json_obj['risk_level']}"

    # Validate predicted_bottleneck
    if not isinstance(json_obj["predicted_bottleneck"], str):
        return False, "predicted_bottleneck not a string"

    if len(json_obj["predicted_bottleneck"]) < 10:
        return False, "predicted_bottleneck too short"

    if len(json_obj["predicted_bottleneck"]) > 500:
        return False, "predicted_bottleneck too long"

    return True, "Valid"

def calculate_tool_f1(predicted, ground_truth):
    """Calculate F1 score for tool prediction"""
    pred_set = set(predicted)
    true_set = set(ground_truth)

    if len(true_set) == 0:
        return 0.0 if len(pred_set) == 0 else 0.0

    tp = len(pred_set & true_set)
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    if precision + recall == 0:
        return 0.0

    f1 = 2 * (precision * recall) / (precision + recall)
    return f1

def evaluate_baseline(test_file="data/test.jsonl", model="llama3-8b-8192"):
    """Comprehensive baseline evaluation"""

    # Load test set
    test_samples = []
    try:
        with open(test_file, "r") as f:
            for line in f:
                if line.strip():
                    test_samples.append(json.loads(line))
    except FileNotFoundError:
        print(f"❌ Test file not found: {test_file}")
        return

    total = len(test_samples)
    print(f"📊 Evaluating baseline model ({model}) on {total} test samples...\n")

    results = ValidationResult()
    results.total_samples = total

    # Run evaluation
    for i, sample in enumerate(test_samples):
        query = sample["input_query"]
        ground_truth = sample["target_json"]

        print(f"[{i+1}/{total}] ", end="", flush=True)

        start_time = time.time()

        try:
            # Get baseline prediction
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": f"Incident: {query}"}
                ],
                temperature=0.0,
                response_format={"type": "json_object"}
            )

            latency = (time.time() - start_time) * 1000  # ms
            results.latencies.append(latency)

            # Try to parse JSON
            try:
                output = json.loads(response.choices[0].message.content)
                results.valid_json += 1
                print("✓", end="", flush=True)
            except json.JSONDecodeError as e:
                results.errors["json_decode_error"] += 1
                print("✗", end="", flush=True)
                results.detailed_results.append({
                    "query": query,
                    "status": "json_decode_error",
                    "error": str(e)
                })
                time.sleep(1)
                continue

            # Validate schema
            is_valid, reason = validate_schema(output)
            if not is_valid:
                results.errors[f"schema_invalid_{reason}"] += 1
                print("✗", end="", flush=True)
                results.detailed_results.append({
                    "query": query,
                    "status": "schema_invalid",
                    "reason": reason,
                    "output": output
                })
                time.sleep(1)
                continue

            results.schema_valid += 1
            print("✓", end="", flush=True)

            # Compare against ground truth
            pred_tools = sorted(output.get("tools_to_call", []))
            true_tools = sorted(ground_truth.get("tools_to_call", []))

            tool_match = pred_tools == true_tools
            if tool_match:
                results.tool_exact_match += 1

            tool_f1 = calculate_tool_f1(
                output.get("tools_to_call", []),
                ground_truth.get("tools_to_call", [])
            )

            # Track per-tool accuracy
            for tool in true_tools:
                results.per_tool_stats[tool]["total"] += 1
                if tool in pred_tools:
                    results.per_tool_stats[tool]["correct"] += 1

            # Risk level comparison
            risk_match = output.get("risk_level") == ground_truth.get("risk_level")
            if risk_match:
                results.risk_level_match += 1

            results.per_risk_stats[ground_truth["risk_level"]]["total"] += 1
            if risk_match:
                results.per_risk_stats[ground_truth["risk_level"]]["correct"] += 1

            # Store detailed result
            results.detailed_results.append({
                "query": query,
                "status": "success",
                "predicted_tools": output.get("tools_to_call", []),
                "true_tools": true_tools,
                "tool_match": tool_match,
                "tool_f1": tool_f1,
                "predicted_risk": output.get("risk_level"),
                "true_risk": ground_truth.get("risk_level"),
                "risk_match": risk_match,
                "latency_ms": latency
            })

            if i % 10 == 0 and i > 0:
                print(f" ({i}/{total})", flush=True)

        except Exception as e:
            results.errors[f"api_error: {type(e).__name__}"] += 1
            print("✗", end="", flush=True)
            results.detailed_results.append({
                "query": query,
                "status": "api_error",
                "error": str(e)
            })

        time.sleep(1)  # Rate limiting

    # Print results
    print_evaluation_report(results)

    # Save detailed results
    save_detailed_results(results)

def print_evaluation_report(results):
    """Print comprehensive evaluation report"""

    print("\n\n" + "="*60)
    print("BASELINE EVALUATION REPORT")
    print("="*60)

    total = results.total_samples

    # Overall metrics
    print("\n📈 OVERALL METRICS")
    print(f"  Total samples: {total}")
    print(f"  Valid JSON: {results.valid_json}/{total} ({results.valid_json/total*100:.1f}%)者に")
    print(f"  Valid schema: {results.schema_valid}/{total} ({results.schema_valid/total*100:.1f}%)者に")

    if results.schema_valid > 0:
        print(f"  Tool exact match: {results.tool_exact_match}/{results.schema_valid} ({results.tool_exact_match/results.schema_valid*100:.1f}%)者に")
        print(f"  Risk level match: {results.risk_level_match}/{results.schema_valid} ({results.risk_level_match/results.schema_valid*100:.1f}%)者に")

    # Latency stats
    if results.latencies:
        print(f"\n⏱️  LATENCY (ms)")
        print(f"  Min: {min(results.latencies):.1f}")
        print(f"  Max: {max(results.latencies):.1f}")
        print(f"  Mean: {statistics.mean(results.latencies):.1f}")
        print(f"  Median: {statistics.median(results.latencies):.1f}")
        print(f"  P99: {sorted(results.latencies)[int(0.99*len(results.latencies))]:.1f}")

    # Per-tool breakdown
    if results.per_tool_stats:
        print(f"\n🔧 PER-TOOL ACCURACY")
        for tool in sorted(results.per_tool_stats.keys()):
            stats = results.per_tool_stats[tool]
            accuracy = stats["correct"] / stats["total"] * 100 if stats["total"] > 0 else 0
            print(f"  {tool}: {stats['correct']}/{stats['total']} ({accuracy:.1f}%)者に")

    # Per-risk breakdown
    if results.per_risk_stats:
        print(f"\n⚠️  PER-RISK-LEVEL ACCURACY")
        for risk in ["high", "medium", "low"]:
            if risk in results.per_risk_stats:
                stats = results.per_risk_stats[risk]
                accuracy = stats["correct"] / stats["total"] * 100 if stats["total"] > 0 else 0
                print(f"  {risk}: {stats['correct']}/{stats['total']} ({accuracy:.1f}%)者に")

    # Error breakdown
    if results.errors:
        print(f"\n❌ ERROR BREAKDOWN")
        for error_type, count in sorted(results.errors.items(), key=lambda x: -x[1]):
            print(f"  {error_type}: {count}")

    print("\n" + "="*60 + "\n")

def save_detailed_results(results):
    """Save detailed results for analysis"""

    output_file = "baseline_evaluation_results.json"

    with open(output_file, "w") as f:
        json.dump({
            "total_samples": results.total_samples,
            "valid_json": results.valid_json,
            "valid_schema": results.schema_valid,
            "tool_exact_match": results.tool_exact_match,
            "risk_level_match": results.risk_level_match,
            "latency_stats": {
                "mean": statistics.mean(results.latencies) if results.latencies else 0,
                "median": statistics.median(results.latencies) if results.latencies else 0,
                "p99": sorted(results.latencies)[int(0.99*len(results.latencies))] if results.latencies else 0
            },
            "per_tool_stats": {
                tool: {
                    "correct": stats["correct"],
                    "total": stats["total"],
                    "accuracy": stats["correct"] / stats["total"] * 100 if stats["total"] > 0 else 0
                }
                for tool, stats in results.per_tool_stats.items()
            },
            "per_risk_stats": {
                risk: {
                    "correct": stats["correct"],
                    "total": stats["total"],
                    "accuracy": stats["correct"] / stats["total"] * 100 if stats["total"] > 0 else 0
                }
                for risk, stats in results.per_risk_stats.items()
            },
            "detailed_results": results.detailed_results
        }, f, indent=2)

    print(f"✅ Detailed results saved to: {output_file}")

if __name__ == "__main__":
    evaluate_baseline(
        test_file="data/test.jsonl",
        model="llama-3.1-8b-instant"  # Baseline model
    )

📊 Evaluating baseline model (llama-3.1-8b-instant) on 20 test samples...

[1/20] ✓✓[2/20] ✓✓[3/20] ✓✓[4/20] ✓✓[5/20] ✓✓[6/20] ✓✓[7/20] ✓✓[8/20] ✓✓[9/20] ✓✓[10/20] ✓✓[11/20] ✓✓ (10/20)
[12/20] ✓✓[13/20] ✓✓[14/20] ✓✓[15/20] ✓✓[16/20] ✓✓[17/20] ✓✓[18/20] ✓✓[19/20] ✓✓[20/20] ✓✓

BASELINE EVALUATION REPORT

📈 OVERALL METRICS
  Total samples: 20
  Valid JSON: 20/20 (100.0%)者に
  Valid schema: 20/20 (100.0%)者に
  Tool exact match: 1/20 (5.0%)者に
  Risk level match: 13/20 (65.0%)者に

⏱️  LATENCY (ms)
  Min: 178.1
  Max: 319.2
  Mean: 226.6
  Median: 221.0
  P99: 319.2

🔧 PER-TOOL ACCURACY
  fetch_network_logs: 11/20 (55.0%)者に
  page_oncall: 0/1 (0.0%)者に
  query_servicenow_db: 10/19 (52.6%)者に
  restart_ec2_node: 3/3 (100.0%)者に
  trigger_anomaly_detection: 7/16 (43.8%)者に

⚠️  PER-RISK-LEVEL ACCURACY
  high: 13/13 (100.0%)者に
  medium: 0/7 (0.0%)者に


✅ Detailed results saved to: baseline_evaluation_results.json


In [ ]:
# Save the model to your Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy the output folder to a persistent Drive location
!cp -r /content/outputs /content/drive/MyDrive/mlops_router_model
print("Model saved to your Google Drive!")

Mounted at /content/drive
cp: cannot stat '/content/outputs': No such file or directory
Model saved to your Google Drive!
